In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Integrated Squared Difference (ISD) Test

This notebook uses the **Integrated Squared Difference (ISD) permutation test** to compare two droplet-freezing curves.

The test measures the accumulated squared separation between the fraction-frozen curves over temperature. It is useful for general comparisons, including curves that differ in shape, cross, or contain different proportions of ice-active populations.

A small p-value provides evidence that the underlying freezing-temperature distributions differ. However, the test does not identify where or why they d p = 0.05 is often used as the threshold for statistical significance.iffer. A non-significant result does not prove that the distributions are identical, particularly when droplet numbers are small.

## Input

The CSV file should contain two columns of freezing temperatures, with the group names as column headings. Blank and non-numeric cells are removed automatically.

## Use

1. Place the CSV file in the same folder as the notebook.
2. Enter its filename in the **Setup** cell.
3. Select the temperature increment and number of permutations.
4. Run all cells in order.
5. Check the group names and droplet numbers.
6. Inspect the p-value, fraction-frozen curves, and permutation distribution.

The shaded regions are pointwise Kaplan–Meier confidence intervals and should not be interpreted as simultaneous confidence bands. The ISD statistic has units of temperature and depends on the analysed temperature interval, although this does not affect the permutation p-value when the same interval is used for every permutation.

In [ ]:
# ============================================================
# Setup
# ============================================================

CSV_FILE = "example_data_tests.csv" # enter input .csv name here. Make sure it is in the same folder as the notebook script.

TEMP_INCREMENT = 0.01 # Resolution of temperature grid
N_PERMUTATIONS = 10000 # More is better but runs slower (shocker). Suggest using 10000
FIGURE_DPI = 300

In [ ]:
# ============================================================
# Create temperature grid
# ============================================================

def create_t_grid(temps1, temps2, temp_increment):
    """create a common temperature axis on which to interpolate each fraction-frozen curve"""


    min_ = min(temps1.min(), temps2.min())
    max_ = max(temps1.max(), temps2.max())
    n = int(round((max_ - min_) / temp_increment)) + 2
    return min_ + temp_increment*(np.arange(n)-1)

In [ ]:
# ============================================================
# Calculate fraction frozen
# ============================================================
def fraction_frozen(temps, t_grid):
    """Compute the fraction-frozen curve on the fine temperature grid. Changed so it sorts data rather than relying on order"""

    temps = np.sort(np.asarray(temps))

    counts = np.searchsorted(temps, t_grid, side="right")
    ff = (len(temps) - counts) / len(temps)

    return ff

In [ ]:
# ============================================================
# Calculate KM Confidence intervals on fraction frozen
# ============================================================

def confidence_bounds(ff_curve, n):
    """KM confidence bounds given no-censorship. If censorship is an issue, refer to the
    larger set of code in the jupyter notebook"""


    s = 1 - ff_curve


    ci_lower = np.full_like(s, np.nan)
    ci_upper = np.full_like(s, np.nan)


    mask = (s > 0) & (s < 1)


    ll_var = (1 - s[mask]) / (n * s[mask] * (np.log(s[mask])**2))


    ci_upper[mask] = 1 - s[mask]**(np.exp( 1.96 * np.sqrt(ll_var)))
    ci_lower[mask] = 1 - s[mask]**(np.exp(-1.96 * np.sqrt(ll_var)))


    ci_lower, ci_upper = fill_ci_edges(ff_curve, ci_lower, ci_upper)


    return ci_lower, ci_upper

def fill_ci_edges(ff, ci_lower, ci_upper):
    """The bounds of the ff curve don't fit the logic of the KM confidence interval.
    Here, we just populate the upper and lower confidence margins with values from the
    neighbouring non-extremal ff values."""


    ci_lower = ci_lower.copy()
    ci_upper = ci_upper.copy()


    #Filling left-hand bits
    idx = np.where(np.isfinite(ci_lower))[0][0]
    fill_val = 1 - (ff[idx] - ci_lower[idx])
    ci_lower[:idx] = fill_val
    ci_upper[:idx] = 1


    #Filling right-hand bits
    idx = np.where(np.isfinite(ci_upper))[0][-1]
    fill_val = ci_upper[idx] - ff[idx]
    ci_upper[(idx+1):] = fill_val
    ci_lower[(idx+1):] = 0


    return ci_lower, ci_upper

In [ ]:
# =================================================================
# Integrated squared difference between two fraction-frozen curves.
# =================================================================

def test_stat(temps1, temps2, t_grid):

    f1 = fraction_frozen(np.sort(temps1), t_grid)
    f2 = fraction_frozen(np.sort(temps2), t_grid)
    dt = np.diff(t_grid)
    return np.sum(((f1[:-1] - f2[:-1]) ** 2) * dt)


In [ ]:
# ====================================================================================
# Combine temperatures and split at random to generate each permutation and test stat
# ====================================================================================

def shuffle(temps1, temps2):  


    temps = np.concatenate((temps1, temps2))
    np.random.shuffle(temps)
    return temps[:len(temps1)], temps[len(temps1):]


In [ ]:
# ========================================================================
# Given a function for computing the test statistic from the temperatures, 
# carry out permutation sampling to build empirical null distribution
# ========================================================================

def create_permutation_distribution(temps1, temps2,
                                    t_grid,
                                    n_monte_carlo):

    q_values = []
    for _ in range(n_monte_carlo):
        t1, t2 = shuffle(temps1, temps2)
        q_values.append(test_stat(t1, t2, t_grid))
    return np.array(q_values)


In [ ]:
# ========================================================================
# Given two sets of freezing temperatures, compute the p-value for the
# observed ISD statistic using permutation testing.
# The null hypothesis assumes that the freezing-temperature distributions
# are identical.
# ========================================================================

def hypothesis_test(
        temps1,
        temps2,
        temp_increment=0.01,
        n_monte_carlo=10_000,
        plot=True):

    # Fine temperature grid
    t_grid = create_t_grid(
        temps1,
        temps2,
        temp_increment)

    # Observed ISD statistic
    q_obs = test_stat(
        temps1,
        temps2,
        t_grid)

    # Empirical null distribution
    q_null_dist = create_permutation_distribution(
        temps1,
        temps2,
        t_grid,
        n_monte_carlo)

    # +1 correction to avoid zero p-values
    n_extreme = np.count_nonzero(q_null_dist >= q_obs)
    p_value = (n_extreme + 1) / (n_monte_carlo + 1)

    print(f"Observed ISD statistic: {q_obs:.4f}")
    print(f"p-value: {p_value:.4f}")

# ===============================================================
# Plot the fraction-frozen curves and ISD permutation distribution
# NOTE: largely written by MS Copilot
# ===============================================================

    if plot:

        ff1 = fraction_frozen(temps1, t_grid)
        ff2 = fraction_frozen(temps2, t_grid)

        cil_1, ciu_1 = confidence_bounds(ff1, len(temps1))
        cil_2, ciu_2 = confidence_bounds(ff2, len(temps2))

        # Create the two-panel figure
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Left panel: fraction-frozen curves
        axes[0].plot(t_grid, ff1, label=COLUMN_1)
        axes[0].fill_between(
            t_grid,
            cil_1,
            ciu_1,
            alpha=0.1)

        axes[0].plot(t_grid, ff2, label=COLUMN_2)
        axes[0].fill_between(
            t_grid,
            cil_2,
            ciu_2,
            alpha=0.1)

        axes[0].set_xlabel("Temperature")
        axes[0].set_ylabel("Fraction frozen")
        axes[0].set_title("Fraction-frozen curves")
        axes[0].legend()

        # Right panel: permutation distribution of ISD
        axes[1].hist(
            q_null_dist,
            bins=40,
            edgecolor="white")

        axes[1].axvline(
            q_obs,
            color="red",
            linestyle="--",
            label=f"Observed Q\np = {p_value:.4f}")

        axes[1].set_xlabel("ISD statistic")
        axes[1].set_ylabel("Count")
        axes[1].set_title("Permutation distribution")
        axes[1].legend()

        plt.tight_layout()

        # Save the figure before displaying it
        fig.savefig(
            FIGURE_FILE,
            dpi=FIGURE_DPI,
            bbox_inches="tight")

        print(f"Figure saved as: {FIGURE_FILE}")

        plt.show()

    return q_obs, p_value

  

In [ ]:
# Read the CSV file
data = pd.read_csv(CSV_FILE)

# Extract the first two column headings
COLUMN_1 = data.columns[0]
COLUMN_2 = data.columns[1]

# Create the output filename after the headings are known
FIGURE_FILE = f"{COLUMN_1}_vs_{COLUMN_2}.png"

# Extract and clean the two columns
temps1 = pd.to_numeric(
    data[COLUMN_1],
    errors="coerce"
).dropna().to_numpy()

temps2 = pd.to_numeric(
    data[COLUMN_2],
    errors="coerce"
).dropna().to_numpy()

# Report what was loaded
print(f"Comparing: {COLUMN_1} and {COLUMN_2}")
print(f"{COLUMN_1}: {len(temps1)} observations")
print(f"{COLUMN_2}: {len(temps2)} observations")
print(f"Figure will be saved as: {FIGURE_FILE}")


In [ ]:
    # Run the hypothesis test
    q_obs, p_value = hypothesis_test(
        temps1,
        temps2,
        temp_increment=TEMP_INCREMENT,
        n_monte_carlo=N_PERMUTATIONS,
        plot = True 
    )